# New Article Keyword Scoping

Use this notebook to test a proposed article topic against related keyword demand in Australia and decide whether the topic should stay broad, split into subtopics, or be renamed.

### Quick start
1. Run the **Setup (run once)** cell.
2. Enter a seed keyword and optional additional seeds.
3. Keep `RUN_MODE="sandbox"` while testing.
4. Review the spend preview before enabling live mode.
5. Run the remaining cells from top to bottom.


In [1]:
#@title Setup (run once)
import os
import re
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd

from lla_data import config
from lla_data.dataforseo import get_user_data, keywords_for_keywords, response_to_keyword_frame


In [ ]:
#@title Parameters
SEED_KEYWORD = "performance anxiety" #@param {type:"string"}
ADDITIONAL_SEED_KEYWORDS = "" #@param {type:"string"}
LOCATION_NAME = config.DEFAULT_KEYWORD_LOCATION #@param {type:"string"}
LANGUAGE_NAME = config.DEFAULT_KEYWORD_LANGUAGE #@param {type:"string"}
RUN_MODE = "live" #@param ["sandbox", "live"]
RUN_LIVE = True #@param {type:"boolean"}
USE_CACHE = True #@param {type:"boolean"}
MIN_SEARCH_VOLUME = 50 #@param {type:"integer"}
MAX_RESULTS = 100 #@param {type:"integer"}

seed_keywords = [SEED_KEYWORD.strip()] + [
    keyword.strip()
    for keyword in ADDITIONAL_SEED_KEYWORDS.split(",")
    if keyword.strip()
]
seed_keywords = list(dict.fromkeys([keyword for keyword in seed_keywords if keyword]))

print(f"Seed keywords: {seed_keywords}")
print(f"DataForSEO market: {LOCATION_NAME} / {LANGUAGE_NAME}")


Seed keywords: ['natural disasters']
DataForSEO market: Australia / English


In [3]:
# Guardrails / spend preview
print("Expected DataForSEO calls for this run: 1 keywords_for_keywords request")
print(f"Run mode: {RUN_MODE}")
print(f"Use cache: {USE_CACHE}")

if RUN_MODE == "live" and not RUN_LIVE:
    raise ValueError("Live DataForSEO calls are blocked. Review the request first, then set RUN_LIVE=True.")

try:
    account = get_user_data(mode="sandbox", use_cache=True)
    account_result = account.payload["tasks"][0]["result"][0]
    print(f"DataForSEO account: {account_result.get('login')}")
    print(f"User-data cache: {account.path}")
except Exception as exc:
    print(f"Could not load DataForSEO account info: {exc}")


Expected DataForSEO calls for this run: 1 keywords_for_keywords request
Run mode: live
Use cache: True
Could not load DataForSEO account info: 404 Client Error: Not Found for url: https://sandbox.dataforseo.com/v3/appendix/user_data


In [4]:
# Related keyword discovery
keyword_response = keywords_for_keywords(
    seed_keywords,
    location_name=LOCATION_NAME,
    language_name=LANGUAGE_NAME,
    mode=RUN_MODE,
    use_cache=USE_CACHE,
    run_live=RUN_LIVE,
)
print(f"Keyword response cache: {keyword_response.path}")
print(f"Loaded from cache: {keyword_response.from_cache}")

related_keywords = response_to_keyword_frame(keyword_response)
related_keywords = related_keywords.dropna(subset=["keyword"]).copy()
related_keywords = related_keywords[
    related_keywords["search_volume"].fillna(0) >= MIN_SEARCH_VOLUME
]
related_keywords = related_keywords.sort_values("search_volume", ascending=False)
related_keywords = related_keywords.head(MAX_RESULTS).reset_index(drop=True)
related_keywords.head(20)


Keyword response cache: /home/aido/projects/lla-data/notebooks/output/dataforseo_v3__keywords_data__google_ads__keywords_for_keywords__live_13c3cc3ce1d4dc6d.json
Loaded from cache: False


,keyword,search_volume,competition,competition_index,cpc,low_top_of_page_bid,high_top_of_page_bid,monthly_searches
0,natural disasters,9900,LOW,0,None,None,None,"[{'year': 2026, 'month': 2, 'search_volume': 5..."


In [5]:
# Topic grouping heuristics for article decisions
seed_tokens = {
    token
    for keyword in seed_keywords
    for token in re.findall(r"[a-z0-9]+", keyword.lower())
    if len(token) > 2
}
question_tokens = {"how", "what", "why", "when", "can", "should", "tips", "help"}

def classify_topic(keyword: str) -> str:
    value = str(keyword).strip().lower()
    tokens = set(re.findall(r"[a-z0-9]+", value))
    if value in {seed.lower() for seed in seed_keywords}:
        return "exact_seed"
    if seed_tokens and seed_tokens.issubset(tokens):
        return "close_variant"
    if tokens & question_tokens:
        return "intent_split"
    if tokens & seed_tokens:
        return "subtopic"
    return "intent_split"

scoped_keywords = related_keywords.copy()
scoped_keywords["topic_group"] = scoped_keywords["keyword"].apply(classify_topic)
scoped_keywords["recommended_action"] = scoped_keywords["topic_group"].map(
    {
        "exact_seed": "Keep as core article target",
        "close_variant": "Keep within same article if SERP intent aligns",
        "subtopic": "Consider section or supporting article",
        "intent_split": "Consider separate article",
    }
)
scoped_keywords.head(30)


,keyword,search_volume,competition,competition_index,cpc,low_top_of_page_bid,high_top_of_page_bid,monthly_searches,topic_group,recommended_action
0,natural disasters,9900,LOW,0,None,None,None,"[{'year': 2026, 'month': 2, 'search_volume': 5...",exact_seed,Keep as core article target


In [6]:
# Decision tables
top_related_keywords = scoped_keywords[
    [
        "keyword",
        "search_volume",
        "competition",
        "competition_index",
        "cpc",
        "topic_group",
        "recommended_action",
        "monthly_searches",
    ]
].copy()

potential_article_splits = (
    scoped_keywords.groupby("topic_group", as_index=False)
    .agg(
        keyword_count=("keyword", "count"),
        total_search_volume=("search_volume", "sum"),
        example_keywords=("keyword", lambda values: ", ".join(list(values)[:5])),
    )
    .sort_values("total_search_volume", ascending=False)
)

low_demand_topics = top_related_keywords[top_related_keywords["search_volume"].fillna(0) < 100].copy()

print("Top related keywords")
display(top_related_keywords.head(30))
print("Potential article splits")
display(potential_article_splits)
print("Low-demand / avoid-as-standalone topics")
display(low_demand_topics.head(30))


Top related keywords


,keyword,search_volume,competition,competition_index,cpc,topic_group,recommended_action,monthly_searches
0,natural disasters,9900,LOW,0,None,exact_seed,Keep as core article target,"[{'year': 2026, 'month': 2, 'search_volume': 5..."


Potential article splits


,topic_group,keyword_count,total_search_volume,example_keywords
0,exact_seed,1,9900,natural disasters


Low-demand / avoid-as-standalone topics


,keyword,search_volume,competition,competition_index,cpc,topic_group,recommended_action,monthly_searches


## How to interpret the tables

- Keep one article when the highest-volume terms cluster tightly around the same intent.
- Split into separate articles when `intent_split` terms carry meaningful volume and look like different user needs.
- Reject the topic as a standalone article when nearly all related demand stays below your practical threshold.
- Rename the article around the stronger phrasing when a close variant clearly carries more demand than the original seed.
